In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict, List, Tuple
import numpy as np

class BayesianLinear(nn.Module):
    """Bayesian Linear layer using reparameterization trick"""
    def __init__(self, in_features: int, out_features: int, prior_std: float = 1.0):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.prior_std = prior_std

        # Mean parameters
        self.weight_mean = nn.Parameter(
            torch.Tensor(out_features, in_features).normal_(0, 0.1)
        )
        self.bias_mean = nn.Parameter(
            torch.Tensor(out_features).normal_(0, 0.1)
        )

        # Log variance parameters (for numerical stability)
        self.weight_logvar = nn.Parameter(
            torch.Tensor(out_features, in_features).normal_(-5, 0.1)
        )
        self.bias_logvar = nn.Parameter(
            torch.Tensor(out_features).normal_(-5, 0.1)
        )

        # Register buffer for precision (inverse variance) tracking
        self.register_buffer(
            'weight_precision',
            torch.ones(out_features, in_features) / (prior_std ** 2)
        )
        self.register_buffer(
            'bias_precision',
            torch.ones(out_features) / (prior_std ** 2)
        )

    def forward(self, x: torch.Tensor, sample: bool = True) -> torch.Tensor:
        """
        Forward pass with optional sampling from weight distribution
        """
        if sample and self.training:
            # Reparameterization trick: w = mean + std * epsilon
            weight_std = torch.exp(0.5 * self.weight_logvar)
            weight_eps = torch.randn_like(weight_std)
            weight = self.weight_mean + weight_std * weight_eps

            bias_std = torch.exp(0.5 * self.bias_logvar)
            bias_eps = torch.randn_like(bias_std)
            bias = self.bias_mean + bias_std * bias_eps
        else:
            # Use mean weights at test time
            weight = self.weight_mean
            bias = self.bias_mean

        return F.linear(x, weight, bias)

    def get_kl_divergence(self) -> torch.Tensor:
        """
        Compute KL divergence between learned and prior distributions
        """
        weight_var = torch.exp(self.weight_logvar)
        bias_var = torch.exp(self.bias_logvar)

        kl_weight = 0.5 * torch.sum(
            self.weight_precision * (weight_var + self.weight_mean ** 2)
            - 1 - self.weight_logvar
        )

        kl_bias = 0.5 * torch.sum(
            self.bias_precision * (bias_var + self.bias_mean ** 2)
            - 1 - self.bias_logvar
        )

        return kl_weight + kl_bias


class BayesianDetectionHead(nn.Module):
    """Bayesian detection head for YOLOv11s"""
    def __init__(self, input_channels: int = 512, num_classes: int = 80):
        super().__init__()
        self.num_classes = num_classes

        # Shared Bayesian layers
        self.fc1 = BayesianLinear(input_channels, 256)
        self.fc2 = BayesianLinear(256, 128)

        # Task-specific heads (initialized but will be task-dependent)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x: torch.Tensor, task: str = 'detection') -> torch.Tensor:
        """
        Forward pass through Bayesian head
        x shape: (batch_size, input_channels)
        """
        x = self.fc1(x, sample=True)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x, sample=True)
        x = self.relu(x)
        x = self.dropout(x)

        return x

    def get_kl_divergence(self) -> torch.Tensor:
        """Sum KL divergences across all Bayesian layers"""
        return self.fc1.get_kl_divergence() + self.fc2.get_kl_divergence()


class B_YOLO(nn.Module):
    """Complete B-YOLO model with task-specific outputs"""
    def __init__(self, backbone: nn.Module, num_classes: int = 80):
        super().__init__()
        self.backbone = backbone  # Frozen YOLOv11s backbone
        self.num_classes = num_classes
        self.bayesian_head = BayesianDetectionHead(512, num_classes)

        # Task-specific output heads
        self.detection_head = nn.Linear(128, 4 + num_classes)  # bbox + class
        self.segmentation_head = nn.Linear(128, 256)  # mask features
        self.pose_head = nn.Linear(128, 17 * 2)  # 17 keypoints * (x, y)
        self.oriented_head = nn.Linear(128, 5 + num_classes)  # bbox + angle + class
        self.classification_head = nn.Linear(128, num_classes)

        self.current_task = None

    def set_task(self, task: str):
        """Set the current task"""
        valid_tasks = ['detection', 'segmentation', 'pose', 'oriented', 'classification']
        assert task in valid_tasks, f"Task must be one of {valid_tasks}"
        self.current_task = task

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Forward pass with task-specific output"""
        # Frozen backbone
        with torch.no_grad():
            backbone_features = self.backbone(x)

        # Adaptive pooling to fixed size
        if backbone_features.dim() == 4:
            backbone_features = F.adaptive_avg_pool2d(backbone_features, (1, 1))
            backbone_features = backbone_features.view(backbone_features.size(0), -1)

        # Bayesian head
        head_features = self.bayesian_head(backbone_features)

        # Task-specific output
        if self.current_task == 'detection':
            output = self.detection_head(head_features)
        elif self.current_task == 'segmentation':
            output = self.segmentation_head(head_features)
        elif self.current_task == 'pose':
            output = self.pose_head(head_features)
        elif self.current_task == 'oriented':
            output = self.oriented_head(head_features)
        elif self.current_task == 'classification':
            output = self.classification_head(head_features)
        else:
            raise ValueError(f"Unknown task: {self.current_task}")

        return output

    def get_kl_divergence(self) -> torch.Tensor:
        """Get total KL divergence from Bayesian layers"""
        return self.bayesian_head.get_kl_divergence()

In [2]:
class TaskSpecificLoss(nn.Module):
    """Task-specific loss functions for B-YOLO"""

    def __init__(self, task: str, num_classes: int = 80, device: torch.device = None):
        super().__init__()
        self.task = task
        self.num_classes = num_classes
        self.device = device or torch.device('cpu')

    def forward(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        kl_divergence: torch.Tensor = None,
        kl_weight: float = 0.01
    ) -> torch.Tensor:
        """
        Compute task-specific loss with optional KL regularization
        """
        if self.task == 'detection':
            task_loss = self._detection_loss(predictions, targets)
        elif self.task == 'segmentation':
            task_loss = self._segmentation_loss(predictions, targets)
        elif self.task == 'pose':
            task_loss = self._pose_loss(predictions, targets)
        elif self.task == 'oriented':
            task_loss = self._oriented_loss(predictions, targets)
        elif self.task == 'classification':
            task_loss = self._classification_loss(predictions, targets)
        else:
            raise ValueError(f"Unknown task: {self.task}")

        # Add KL divergence regularization
        if kl_divergence is not None:
            total_loss = task_loss + kl_weight * kl_divergence
        else:
            total_loss = task_loss

        return total_loss

    def _detection_loss(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:
        """
        Detection loss: bounding box regression + classification
        predictions: (batch, 4 + num_classes)
        targets: (batch, 4 + num_classes) with 4 bbox coords + class one-hot
        """
        batch_size = predictions.size(0)

        # Bounding box loss (smooth L1)
        bbox_pred = predictions[:, :4]
        bbox_target = targets[:, :4]
        bbox_loss = F.smooth_l1_loss(bbox_pred, bbox_target, reduction='mean')

        # Classification loss (cross entropy)
        class_pred = predictions[:, 4:]
        class_target = targets[:, 4:].argmax(dim=1)
        class_loss = F.cross_entropy(class_pred, class_target, reduction='mean')

        return 0.5 * bbox_loss + class_loss

    def _segmentation_loss(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:
        """
        Segmentation loss: binary cross entropy for pixel-level masks
        predictions: (batch, 256) feature map
        targets: (batch, 256) binary mask
        """
        # Reshape for BCE
        mask_loss = F.binary_cross_entropy_with_logits(
            predictions,
            targets,
            reduction='mean'
        )

        # Dice loss for better segmentation
        predictions_sigmoid = torch.sigmoid(predictions)
        intersection = (predictions_sigmoid * targets).sum()
        dice = 1 - (2 * intersection) / (predictions_sigmoid.sum() + targets.sum() + 1e-8)

        return 0.5 * mask_loss + 0.5 * dice

    def _pose_loss(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:
        """
        Pose loss: keypoint localization with confidence
        predictions: (batch, 34) [17 keypoints * (x, y)]
        targets: (batch, 34) [17 keypoints * (x, y)]
        """
        # MSE loss for keypoint coordinates
        keypoint_loss = F.mse_loss(predictions, targets, reduction='mean')

        # Smooth L1 for robustness
        smooth_loss = F.smooth_l1_loss(predictions, targets, reduction='mean')

        return 0.7 * keypoint_loss + 0.3 * smooth_loss

    def _oriented_loss(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:
        """
        Oriented detection loss: bbox + rotation angle + classification
        predictions: (batch, 5 + num_classes) [4 bbox + angle + class]
        targets: (batch, 5 + num_classes)
        """
        # Bounding box loss
        bbox_pred = predictions[:, :4]
        bbox_target = targets[:, :4]
        bbox_loss = F.smooth_l1_loss(bbox_pred, bbox_target, reduction='mean')

        # Angle loss (circular regression)
        angle_pred = predictions[:, 4:5]
        angle_target = targets[:, 4:5]
        # Normalize angles to [-pi, pi]
        angle_diff = torch.atan2(
            torch.sin(angle_pred - angle_target),
            torch.cos(angle_pred - angle_target)
        )
        angle_loss = F.mse_loss(angle_diff, torch.zeros_like(angle_diff), reduction='mean')

        # Classification loss
        class_pred = predictions[:, 5:]
        class_target = targets[:, 5:].argmax(dim=1)
        class_loss = F.cross_entropy(class_pred, class_target, reduction='mean')

        return 0.4 * bbox_loss + 0.3 * angle_loss + 0.3 * class_loss

    def _classification_loss(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor
    ) -> torch.Tensor:
        """
        Image classification loss: standard cross entropy
        predictions: (batch, num_classes)
        targets: (batch, num_classes) one-hot or (batch,) class indices
        """
        if targets.dim() == 2:
            # One-hot encoded targets
            class_loss = F.cross_entropy(
                predictions,
                targets.argmax(dim=1),
                reduction='mean'
            )
        else:
            # Direct class indices
            class_loss = F.cross_entropy(predictions, targets, reduction='mean')

        return class_loss


class EWCRegularizer(nn.Module):
    """Elastic Weight Consolidation for continual learning"""

    def __init__(self, model: B_YOLO, lambda_ewc: float = 0.4):
        super().__init__()
        self.model = model
        self.lambda_ewc = lambda_ewc
        self.fisher_information = None
        self.previous_weights = None

    def compute_fisher_information(
        self,
        dataloader: torch.utils.data.DataLoader,
        device: torch.device
    ):
        """
        Compute Fisher Information Matrix for current task
        """
        self.model.eval()
        fisher = {}

        for name, param in self.model.named_parameters():
            if 'bayesian' in name:
                fisher[name] = torch.zeros_like(param.data)

        for batch_idx, (images, _) in enumerate(dataloader):
            images = images.to(device)

            self.model.eval()
            outputs = self.model(images)

            # Use output entropy as proxy for task loss
            loss = torch.sum(torch.log(outputs + 1e-8))

            self.model.zero_grad()
            loss.backward(retain_graph=True)

            for name, param in self.model.named_parameters():
                if 'bayesian' in name and param.grad is not None:
                    fisher[name] += param.grad.data.pow(2)

        # Average over batches
        for name in fisher:
            fisher[name] /= (batch_idx + 1)
            # Convert to precision (Fisher approximates Hessian)
            fisher[name] = fisher[name] + 1e-8

        self.fisher_information = fisher
        self._save_previous_weights()

    def _save_previous_weights(self):
        """Store current weights as reference"""
        self.previous_weights = {}
        for name, param in self.model.named_parameters():
            if 'bayesian' in name:
                self.previous_weights[name] = param.data.clone()

    def get_ewc_loss(self) -> torch.Tensor:
        """
        Compute EWC regularization loss
        L_EWC = (lambda/2) * sum(F_i * (w_i - w_old_i)^2)
        """
        if self.fisher_information is None or self.previous_weights is None:
            return torch.tensor(0.0)

        ewc_loss = torch.tensor(0.0, device=self.model.bayesian_head.fc1.weight_mean.device)

        for name, param in self.model.named_parameters():
            if name in self.fisher_information:
                fisher = self.fisher_information[name]
                prev_weight = self.previous_weights[name]
                ewc_loss += torch.sum(fisher * (param - prev_weight) ** 2)

        return (self.lambda_ewc / 2) * ewc_loss

In [3]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import os
from pathlib import Path
from tqdm import tqdm
from collections import defaultdict

class B_YOLOTrainer:
    """Complete training loop for B-YOLO with continual learning"""

    def __init__(
        self,
        model: B_YOLO,
        tasks: List[str],
        device: torch.device = None,
        checkpoint_dir: str = './checkpoints'
    ):
        self.model = model
        self.tasks = tasks
        self.device = device or torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.checkpoint_dir = Path(checkpoint_dir)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)

        self.ewc_regularizer = EWCRegularizer(model)
        self.best_metrics = defaultdict(lambda: {'mAP': 0.0, 'model_path': None})
        self.training_history = defaultdict(list)

    def train_sequential_tasks(
        self,
        task_dataloaders: Dict[str, Dict[str, DataLoader]],
        num_epochs_per_task: int = 50,
        learning_rate: float = 1e-3,
        kl_weight: float = 0.01,
        lambda_ewc: float = 0.4
    ):
        """
        Train model sequentially on multiple tasks
        task_dataloaders: {task_name: {'train': DataLoader, 'val': DataLoader}}
        """
        self.model = self.model.to(self.device)

        for task_idx, task in enumerate(self.tasks):
            print(f"\n{'='*60}")
            print(f"Task {task_idx + 1}/{len(self.tasks)}: {task.upper()}")
            print(f"{'='*60}")

            self.model.set_task(task)
            loss_fn = TaskSpecificLoss(task, self.model.num_classes, self.device)

            optimizer = optim.Adam(
                filter(lambda p: p.requires_grad, self.model.parameters()),
                lr=learning_rate
            )
            scheduler = optim.lr_scheduler.ReduceLROnPlateau(
                optimizer, mode='max', factor=0.5, patience=5, verbose=True
            )

            train_loader = task_dataloaders[task]['train']
            val_loader = task_dataloaders[task]['val']

            best_val_metric = 0.0
            patience_counter = 0
            patience = 10

            for epoch in range(num_epochs_per_task):
                # Training phase
                train_loss = self._train_epoch(
                    train_loader,
                    task,
                    loss_fn,
                    optimizer,
                    kl_weight,
                    lambda_ewc if task_idx > 0 else 0.0
                )

                # Validation phase
                val_metric = self._validate(val_loader, task, loss_fn)

                self.training_history[task].append({
                    'epoch': epoch,
                    'train_loss': train_loss,
                    'val_metric': val_metric
                })

                print(f"Epoch {epoch + 1}/{num_epochs_per_task} | "
                      f"Train Loss: {train_loss:.4f} | "
                      f"Val Metric: {val_metric:.4f}")

                # Save best model
                if val_metric > best_val_metric:
                    best_val_metric = val_metric
                    patience_counter = 0
                    self._save_checkpoint(task, epoch, val_metric)
                else:
                    patience_counter += 1

                scheduler.step(val_metric)

                # Early stopping
                if patience_counter >= patience:
                    print(f"Early stopping after {patience} epochs without improvement")
                    break

            # Compute Fisher Information for next task
            if task_idx < len(self.tasks) - 1:
                print(f"\nComputing Fisher Information for EWC...")
                self.ewc_regularizer.compute_fisher_information(
                    train_loader,
                    self.device
                )

    def _train_epoch(
        self,
        dataloader: DataLoader,
        task: str,
        loss_fn: TaskSpecificLoss,
        optimizer: optim.Optimizer,
        kl_weight: float,
        ewc_weight: float
    ) -> float:
        """Single training epoch"""
        self.model.train()
        total_loss = 0.0
        num_batches = 0

        pbar = tqdm(dataloader, desc=f"Training {task}", leave=False)

        for images, targets in pbar:
            images = images.to(self.device)
            targets = targets.to(self.device)

            optimizer.zero_grad()

            # Forward pass
            predictions = self.model(images)

            # Task loss
            kl_div = self.model.get_kl_divergence()
            task_loss = loss_fn(predictions, targets, kl_div, kl_weight)

            # EWC regularization
            ewc_loss = self.ewc_regularizer.get_ewc_loss()

            # Total loss
            total_loss_batch = task_loss + ewc_weight * ewc_loss

            # Backward pass
            total_loss_batch.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            optimizer.step()

            total_loss += total_loss_batch.item()
            num_batches += 1
            pbar.update(1)

        return total_loss / num_batches

    def _validate(
        self,
        dataloader: DataLoader,
        task: str,
        loss_fn: TaskSpecificLoss
    ) -> float:
        """Validation phase - returns task-specific metric"""
        self.model.eval()
        total_metric = 0.0
        num_batches = 0

        with torch.no_grad():
            for images, targets in dataloader:
                images = images.to(self.device)
                targets = targets.to(self.device)

                predictions = self.model(images)

                # Compute metric based on task
                metric = self._compute_task_metric(
                    predictions, targets, task
                )
                total_metric += metric
                num_batches += 1

        return total_metric / num_batches if num_batches > 0 else 0.0

    def _compute_task_metric(
        self,
        predictions: torch.Tensor,
        targets: torch.Tensor,
        task: str
    ) -> float:
        """Compute task-specific validation metric"""
        if task == 'detection' or task == 'oriented':
            # mAP approximation: use IoU for bboxes
            pred_boxes = predictions[:, :4]
            target_boxes = targets[:, :4]
            iou = self._compute_iou(pred_boxes, target_boxes)
            return iou.mean().item()

        elif task == 'segmentation':
            # Dice coefficient
            pred_masks = torch.sigmoid(predictions)
            intersection = (pred_masks * targets).sum()
            dice = (2 * intersection) / (pred_masks.sum() + targets.sum() + 1e-8)
            return dice.item()

        elif task == 'pose':
            # Mean Euclidean distance (lower is better, so negate)
            distance = torch.norm(predictions - targets, p=2, dim=1).mean()
            return -distance.item()

        elif task == 'classification':
            # Top-1 accuracy
            pred_classes = predictions.argmax(dim=1)
            target_classes = targets.argmax(dim=1) if targets.dim() == 2 else targets
            accuracy = (pred_classes == target_classes).float().mean()
            return accuracy.item()

        return 0.0

    def _compute_iou(
        self,
        pred_boxes: torch.Tensor,
        target_boxes: torch.Tensor
    ) -> torch.Tensor:
        """
        Compute IoU between predicted and target bounding boxes
        boxes format: (x1, y1, x2, y2)
        """
        # Intersection
        x1_inter = torch.max(pred_boxes[:, 0], target_boxes[:, 0])
        y1_inter = torch.max(pred_boxes[:, 1], target_boxes[:, 1])
        x2_inter = torch.min(pred_boxes[:, 2], target_boxes[:, 2])
        y2_inter = torch.min(pred_boxes[:, 3], target_boxes[:, 3])

        inter_width = (x2_inter - x1_inter).clamp(min=0)
        inter_height = (y2_inter - y1_inter).clamp(min=0)
        intersection = inter_width * inter_height

        # Union
        pred_area = (pred_boxes[:, 2] - pred_boxes[:, 0]) * \
                    (pred_boxes[:, 3] - pred_boxes[:, 1])
        target_area = (target_boxes[:, 2] - target_boxes[:, 0]) * \
                      (target_boxes[:, 3] - target_boxes[:, 1])
        union = pred_area + target_area - intersection

        iou = intersection / (union + 1e-8)
        return iou

    def _save_checkpoint(self, task: str, epoch: int, metric: float):
        """Save best model checkpoint for task"""
        checkpoint_path = self.checkpoint_dir / f"b_yolo_{task}_best.pt"

        torch.save({
            'epoch': epoch,
            'model_state_dict': self.model.state_dict(),
            'metric': metric,
            'task': task,
            'fisher_information': self.ewc_regularizer.fisher_information,
            'previous_weights': self.ewc_regularizer.previous_weights
        }, checkpoint_path)

        self.best_metrics[task]['mAP'] = metric
        self.best_metrics[task]['model_path'] = str(checkpoint_path)

        print(f" Saved best {task} model (metric: {metric:.4f})")

    def load_checkpoint(self, task: str) -> bool:
        """Load best model for specific task"""
        if task not in self.best_metrics:
            print(f"No checkpoint found for task: {task}")
            return False

        checkpoint_path = self.best_metrics[task]['model_path']
        if not os.path.exists(checkpoint_path):
            print(f"Checkpoint file not found: {checkpoint_path}")
            return False

        checkpoint = torch.load(checkpoint_path, map_location=self.device)
        self.model.load_state_dict(checkpoint['model_state_dict'])

        # Restore Fisher Information if available
        if 'fisher_information' in checkpoint:
            self.ewc_regularizer.fisher_information = checkpoint['fisher_information']
            self.ewc_regularizer.previous_weights = checkpoint['previous_weights']

        print(f" Loaded {task} model from {checkpoint_path}")
        return True

    def print_summary(self):
        """Print training summary and best metrics"""
        print(f"\n{'='*60}")
        print("TRAINING SUMMARY")
        print(f"{'='*60}")

        for task in self.tasks:
            if task in self.best_metrics:
                metric = self.best_metrics[task]['mAP']
                print(f"{task.upper():20s} | Best Metric: {metric:.4f}")

In [6]:
!pip install torchinfo
from torchinfo import summary

# Dummy backbone for B_YOLO model
class DummyBackbone(nn.Module):
    def forward(self, x):
        return torch.randn(x.size(0), 512)

dummy_backbone = DummyBackbone()

# Instantiate the B_YOLO model
model = B_YOLO(backbone=dummy_backbone, num_classes=80)

dummy_input = torch.randn(2, 3, 640, 640)
model.set_task('detection')
summary(model, input_data=dummy_input)

Layer (type:depth-idx)                   Output Shape              Param #
B_YOLO                                   [2, 84]                   58,695
├─DummyBackbone: 1-1                     [2, 512]                  --
├─BayesianDetectionHead: 1-2             [2, 128]                  --
│    └─BayesianLinear: 2-1               [2, 256]                  262,656
│    └─ReLU: 2-2                         [2, 256]                  --
│    └─Dropout: 2-3                      [2, 256]                  --
│    └─BayesianLinear: 2-4               [2, 128]                  65,792
│    └─ReLU: 2-5                         [2, 128]                  --
│    └─Dropout: 2-6                      [2, 128]                  --
├─Linear: 1-3                            [2, 84]                   10,836
Total params: 397,979
Trainable params: 397,979
Non-trainable params: 0
Total mult-adds (Units.MEGABYTES): 151.34
Input size (MB): 9.83
Forward/backward pass size (MB): 0.01
Params size (MB): 1.36
Estimated T